# Chapter 2 — The Embedding Layer (`encoder_forward` / `encoder_backward`)

> Course: **llm.c — Zero to Hero**, Chapter 2 of ~20.
> Builds on Chapter 1 (tensors as 1D arrays, the row-pointer trick).

In Chapter 1 you learned that a tensor in `llm.c` is just a `float*` whose shape lives in your variables. Now we'll use that machinery to build the **very first layer of GPT-2**: the embedding layer.

This is a layer you've used a thousand times in PyTorch (`nn.Embedding`) and probably never thought about. In `llm.c` it is laid bare, and in only ~20 lines of C it teaches us:

- That embeddings are not a "neural network operation" — they're a **gather** (an indexed memory read).
- That gradients of the embedding tables are a **scatter-add** — and the "add" matters.
- The first non-trivial backward pass to reason about.
- Our first encounter with **parallelism**: why the forward is trivially parallelizable but a naive parallel backward has a **race condition**.

### Learning objectives

By the end of this chapter you will:

- Read and write `encoder_forward` from scratch in C.
- Read and write `encoder_backward`, and explain *why* it uses `+=` and not `=`.
- Understand the difference between a **gather** (forward) and a **scatter-add** (backward).
- Add an OpenMP `#pragma` to a safe forward loop and identify the race in the naive backward.


## 1. The Concept — Embedding as a Lookup

GPT-2's input is a sequence of token ids: integers in `[0, V)` where `V` is the vocabulary size (50257 for GPT-2). Before the model can do any math on them, it must turn each integer into a `C`-dimensional vector.

The model keeps two learnable tables:

| Table | Shape  | Meaning |
|-------|--------|---------|
| `wte` | `(V, C)`     | row `i` is the embedding of **token id** `i` (Weight Token Embedding) |
| `wpe` | `(maxT, C)`  | row `t` is the embedding of **position** `t` (Weight Position Embedding) |

For a batch of token ids `inp` of shape `(B, T)`, the output of the embedding layer is:

$$\text{out}[b, t, :] = \text{wte}[\text{inp}[b,t], :] + \text{wpe}[t, :]$$

That is it. Two table lookups and an elementwise add. No matrix multiply, no activation. **The layer's "computation" is mostly memory addressing.**

Why two tables? The Transformer architecture has no built-in notion of order — if you shuffle the tokens, attention gives the same outputs. Adding `wpe[t]` injects the position so the model knows *where* each token is in the sequence.


## 2. PyTorch Baseline

In PyTorch you'd write the layer like this:


In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)
B, T, C, V, maxT = 2, 4, 8, 10, 16

wte = nn.Embedding(V, C)        # token embeddings: (V, C)
wpe = nn.Embedding(maxT, C)     # position embeddings: (maxT, C)

inp = torch.randint(0, V, (B, T))  # (B, T) of token ids
positions = torch.arange(T)        # (T,) of positions 0..T-1

out = wte(inp) + wpe(positions)    # (B, T, C)
print("inp:", inp)
print("out shape:", out.shape)
print("out[0, 0, :3]:", out[0, 0, :3].tolist())


Three things to notice:

1. `nn.Embedding(V, C)` is **just a `(V, C)` weight matrix** with a `forward` that does `weight[idx]`. There is no nonlinearity, no bias.
2. `wpe(positions)` broadcasts a `(T, C)` over the batch dimension automatically. The C version does the addition explicitly per `(b, t)` pair.
3. PyTorch's autograd will route gradients back into the embedding tables when we call `.backward()`. We will write that gradient routing by hand.


## 3. The C Forward Pass

Here is the full `encoder_forward` from [`train_gpt2.c`](train_gpt2.c) (lines 35–58), copied verbatim:

```c
void encoder_forward(float* out,
                   int* inp, float* wte, float* wpe,
                   int B, int T, int C) {
    // out is (B,T,C). At each position (b,t), a C-dimensional vector summarizing token & position
    // inp is (B,T) of integers, holding the token ids at each (b,t) position
    // wte is (V,C) of token embeddings, short for "weight token embeddings"
    // wpe is (maxT,C) of position embeddings, short for "weight positional embedding"
    for (int b = 0; b < B; b++) {
        for (int t = 0; t < T; t++) {
            // seek to the output position in out[b,t,:]
            float* out_bt = out + b * T * C + t * C;
            // get the index of the token at inp[b, t]
            int ix = inp[b * T + t];
            // seek to the position in wte corresponding to the token
            float* wte_ix = wte + ix * C;
            // seek to the position in wpe corresponding to the position
            float* wpe_t = wpe + t * C;
            // add the two vectors and store the result in out[b,t,:]
            for (int i = 0; i < C; i++) {
                out_bt[i] = wte_ix[i] + wpe_t[i];
            }
        }
    }
}
```

Read it line by line:

| Line | What it does |
|------|--------------|
| `for (int b = 0; b < B; ...)` | Loop over each item in the batch. |
| `for (int t = 0; t < T; ...)` | Loop over each token position in the sequence. |
| `float* out_bt = out + b*T*C + t*C;` | **Row pointer** to `out[b, t, :]` — Chapter 1 idiom. |
| `int ix = inp[b*T + t];` | Read the scalar at `inp[b, t]`. Note `inp` is `(B,T)` so stride is just `T`. |
| `float* wte_ix = wte + ix*C;` | **Gather**: pointer to row `ix` of `wte`. This is the embedding table lookup. |
| `float* wpe_t = wpe + t*C;` | Pointer to row `t` of `wpe`. |
| `out_bt[i] = wte_ix[i] + wpe_t[i];` | Elementwise sum into the output row. |

Notice how `nn.Embedding`'s entire "forward pass" reduces to one pointer expression: `wte + ix * C`. That is the *whole* layer.


## 4. The Translation Bridge

| PyTorch                                | C in `llm.c` |
|---|---|
| `nn.Embedding(V, C)` (the module)      | A bare `float* wte` of size `V*C`. No "module" — just the weight. |
| `wte(idx)` for scalar `idx`            | `wte + idx * C` (a pointer to the row) |
| `wte(inp)` for a `(B,T)` tensor `inp`  | The double `for (b)` `for (t)` loop above |
| `output = wte(inp) + wpe(positions)`   | `out_bt[i] = wte_ix[i] + wpe_t[i];` (explicit elementwise) |
| Broadcasting `wpe[(T,C)]` over batch  | C ignores broadcasting — it just visits `wpe_t` once per `(b,t)`. The reload is free thanks to CPU caches. |
| Autograd builds the grad graph for you | You write `encoder_backward` by hand. |

Mental model upgrade from Chapter 1:

> An embedding "lookup" in C is **just `wte + ix * C`** — the address of row `ix`. Indexing a tensor by an integer is identical to *advancing a pointer by that many rows*.


## 5. Running `encoder_forward` and Checking it Against PyTorch

Let's actually compile and run `encoder_forward`, then compare numerically to PyTorch.


In [ ]:
!mkdir -p course/ch02_build


In [ ]:
%%writefile course/ch02_build/encoder_forward.c
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

// Lifted verbatim from train_gpt2.c (lines 35-58)
void encoder_forward(float* out,
                     int* inp, float* wte, float* wpe,
                     int B, int T, int C) {
    for (int b = 0; b < B; b++) {
        for (int t = 0; t < T; t++) {
            float* out_bt = out + b * T * C + t * C;
            int ix = inp[b * T + t];
            float* wte_ix = wte + ix * C;
            float* wpe_t = wpe + t * C;
            for (int i = 0; i < C; i++) {
                out_bt[i] = wte_ix[i] + wpe_t[i];
            }
        }
    }
}

// Read a binary file into a freshly malloc'd buffer
static void* read_file(const char* path, size_t nbytes) {
    FILE* f = fopen(path, "rb");
    if (!f) { fprintf(stderr, "open %s failed\n", path); exit(1); }
    void* buf = malloc(nbytes);
    size_t n = fread(buf, 1, nbytes, f);
    if (n != nbytes) { fprintf(stderr, "short read\n"); exit(1); }
    fclose(f);
    return buf;
}

int main(int argc, char** argv) {
    if (argc != 6) {
        fprintf(stderr, "usage: %s B T C V maxT\n", argv[0]); return 1;
    }
    int B = atoi(argv[1]), T = atoi(argv[2]), C = atoi(argv[3]);
    int V = atoi(argv[4]), maxT = atoi(argv[5]);

    int*   inp = (int*)   read_file("course/ch02_build/inp.bin",  (size_t)B*T*sizeof(int));
    float* wte = (float*) read_file("course/ch02_build/wte.bin",  (size_t)V*C*sizeof(float));
    float* wpe = (float*) read_file("course/ch02_build/wpe.bin",  (size_t)maxT*C*sizeof(float));
    float* out = (float*) malloc((size_t)B*T*C*sizeof(float));

    encoder_forward(out, inp, wte, wpe, B, T, C);

    // Write the output for Python to load
    FILE* f = fopen("course/ch02_build/out.bin", "wb");
    fwrite(out, sizeof(float), (size_t)B*T*C, f);
    fclose(f);

    free(inp); free(wte); free(wpe); free(out);
    return 0;
}


In [ ]:
# Compile our C reference
!gcc -O2 -o course/ch02_build/encoder_forward course/ch02_build/encoder_forward.c


In [ ]:
# Generate inputs in Python, save to .bin, run the C binary, load result, compare to PyTorch
import numpy as np, torch, subprocess, os

torch.manual_seed(0)
B, T, C, V, maxT = 2, 4, 8, 10, 16

inp = torch.randint(0, V, (B, T), dtype=torch.int32)
wte = torch.randn(V, C)
wpe = torch.randn(maxT, C)

os.makedirs("course/ch02_build", exist_ok=True)
inp.numpy().astype(np.int32).tofile("course/ch02_build/inp.bin")
wte.numpy().astype(np.float32).tofile("course/ch02_build/wte.bin")
wpe.numpy().astype(np.float32).tofile("course/ch02_build/wpe.bin")

subprocess.run(["./course/ch02_build/encoder_forward",
                str(B), str(T), str(C), str(V), str(maxT)], check=True)

out_c = np.fromfile("course/ch02_build/out.bin", dtype=np.float32).reshape(B, T, C)
out_pt = (wte[inp.long()] + wpe[torch.arange(T)]).numpy()

max_err = np.max(np.abs(out_c - out_pt))
print(f"Max abs diff between C and PyTorch: {max_err:.2e}")
assert max_err < 1e-6, "C and PyTorch disagree!"
print("Match. Our C encoder_forward is bit-for-bit equivalent to nn.Embedding+nn.Embedding+add.")


That `1e-6` tolerance is just float32 noise — the result agrees to the last bit. **You just ran a layer of GPT-2 in C.**


## 6. The Backward Pass — Math First

PyTorch handles the backward automatically; we will do it by hand. The forward at one position is:

$$\text{out}_{b,t,i} = \text{wte}_{ix,i} + \text{wpe}_{t,i} \quad \text{where } ix = \text{inp}[b,t]$$

We are given an upstream gradient `dout` of the same shape as `out`. By the chain rule:

$$\frac{\partial \text{out}_{b,t,i}}{\partial \text{wte}_{ix,i}} = 1, \qquad \frac{\partial \text{out}_{b,t,i}}{\partial \text{wpe}_{t,i}} = 1$$

So the gradient with respect to each table is just *the upstream gradient routed back to the right row*:

$$\text{dwte}[ix, i] \mathrel{+}= \text{dout}[b, t, i]$$
$$\text{dwpe}[t, i]  \mathrel{+}= \text{dout}[b, t, i]$$

Why **`+=`** and not **`=`**? Because the same row of each table is touched by *many* `(b, t)` positions:

- `wte[ix]` is used wherever the token id `ix` appears in the batch — a common token (e.g. " the") may appear dozens of times.
- `wpe[t]` is used by *every* batch element at position `t` — so each `t` is hit `B` times.

A forward "use a row once" turns into a backward "rows accumulate gradient from every place they were used". This pattern is called **scatter-add**, and it's the dual of the forward's **gather**.


## 7. The C Backward Pass

From [`train_gpt2.c`](train_gpt2.c) lines 60–76:

```c
void encoder_backward(float* dwte, float* dwpe,
                      float* dout, int* inp,
                      int B, int T, int C) {
    for (int b = 0; b < B; b++) {
        for (int t = 0; t < T; t++) {
            float* dout_bt = dout + b * T * C + t * C;
            int ix = inp[b * T + t];
            float* dwte_ix = dwte + ix * C;
            float* dwpe_t = dwpe + t * C;
            for (int i = 0; i < C; i++) {
                float d = dout_bt[i];
                dwte_ix[i] += d;
                dwpe_t[i] += d;
            }
        }
    }
}
```

Notice three things:

1. **Same row-pointer pattern as the forward.** `dwte_ix = dwte + ix*C`, `dwpe_t = dwpe + t*C`. These are pointers into the **gradient buffers**, not the parameter buffers.
2. **`+=` not `=`** — for the reason explained above.
3. **`dwte` and `dwpe` are *assumed to be zeroed beforehand*.** `llm.c` does this explicitly via `gpt2_zero_grad` (we'll see this in Chapter 8). If you forgot to zero them, every backward call would accumulate on top of stale gradients.


## 8. Numerical Check Against PyTorch Autograd

Let's confirm our hand-written backward matches PyTorch's autograd byte-for-byte.


In [ ]:
%%writefile course/ch02_build/encoder_backward.c
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

void encoder_backward(float* dwte, float* dwpe,
                      float* dout, int* inp,
                      int B, int T, int C) {
    for (int b = 0; b < B; b++) {
        for (int t = 0; t < T; t++) {
            float* dout_bt = dout + b * T * C + t * C;
            int ix = inp[b * T + t];
            float* dwte_ix = dwte + ix * C;
            float* dwpe_t = dwpe + t * C;
            for (int i = 0; i < C; i++) {
                float d = dout_bt[i];
                dwte_ix[i] += d;
                dwpe_t[i] += d;
            }
        }
    }
}

static void* read_file(const char* path, size_t n) {
    FILE* f = fopen(path, "rb"); if (!f){perror(path); exit(1);}
    void* b = malloc(n); size_t r = fread(b,1,n,f); (void)r; fclose(f); return b;
}

int main(int argc, char** argv) {
    if (argc != 6) { fprintf(stderr,"usage: B T C V maxT\n"); return 1; }
    int B=atoi(argv[1]), T=atoi(argv[2]), C=atoi(argv[3]), V=atoi(argv[4]), maxT=atoi(argv[5]);

    int*   inp  = (int*)   read_file("course/ch02_build/inp.bin",  (size_t)B*T*sizeof(int));
    float* dout = (float*) read_file("course/ch02_build/dout.bin", (size_t)B*T*C*sizeof(float));

    // CRITICAL: zero the gradient buffers before accumulating
    float* dwte = (float*) calloc((size_t)V*C, sizeof(float));
    float* dwpe = (float*) calloc((size_t)maxT*C, sizeof(float));

    encoder_backward(dwte, dwpe, dout, inp, B, T, C);

    FILE* f1 = fopen("course/ch02_build/dwte.bin","wb"); fwrite(dwte,4,(size_t)V*C,f1); fclose(f1);
    FILE* f2 = fopen("course/ch02_build/dwpe.bin","wb"); fwrite(dwpe,4,(size_t)maxT*C,f2); fclose(f2);

    free(inp); free(dout); free(dwte); free(dwpe);
    return 0;
}


In [ ]:
!gcc -O2 -o course/ch02_build/encoder_backward course/ch02_build/encoder_backward.c


In [ ]:
# Run PyTorch autograd and compare
import numpy as np, torch, subprocess

torch.manual_seed(0)
B, T, C, V, maxT = 2, 4, 8, 10, 16

inp_t = torch.randint(0, V, (B, T))
wte_t = torch.randn(V, C, requires_grad=True)
wpe_t = torch.randn(maxT, C, requires_grad=True)

# Forward
out = wte_t[inp_t] + wpe_t[torch.arange(T)]
# Some upstream gradient (would normally come from the next layer's backward)
dout = torch.randn_like(out)

# Backward via autograd
out.backward(dout)

# Save inputs for the C binary
inp_t.numpy().astype(np.int32).tofile("course/ch02_build/inp.bin")
dout.detach().numpy().astype(np.float32).tofile("course/ch02_build/dout.bin")

subprocess.run(["./course/ch02_build/encoder_backward",
                str(B), str(T), str(C), str(V), str(maxT)], check=True)

dwte_c = np.fromfile("course/ch02_build/dwte.bin", dtype=np.float32).reshape(V, C)
dwpe_c = np.fromfile("course/ch02_build/dwpe.bin", dtype=np.float32).reshape(maxT, C)

print(f"max |dwte_c - dwte_pt|: {np.max(np.abs(dwte_c - wte_t.grad.numpy())):.2e}")
print(f"max |dwpe_c - dwpe_pt|: {np.max(np.abs(dwpe_c - wpe_t.grad.numpy())):.2e}")

# Show the per-token-row gradient norms — common tokens have larger grads
# because they appear at multiple (b,t) positions and accumulate.
import collections
token_counts = collections.Counter(inp_t.flatten().tolist())
print("\nPer-row dwte L2 norm vs how often that token appeared:")
for ix in range(V):
    print(f"  token {ix}: count={token_counts.get(ix,0)}, ||dwte[{ix}]||={np.linalg.norm(dwte_c[ix]):.3f}")


Look at that final table. Tokens that appeared more often (`count` higher) have **larger** `dwte` rows — concrete evidence of the scatter-add accumulation. Tokens that didn't appear at all have *exactly zero* gradient.

This is the **first place autograd's "magic" stops feeling magic**. It's just routing gradient back to the rows that were read in the forward pass.


## 9. Toy Example — A Tiny Hand-Traceable Backward

Let's run a minimal example you can verify with pencil and paper.

- `B=1, T=3, V=3, C=2`
- Input tokens: `[0, 1, 0]` — token `0` appears **twice**, token `1` once, token `2` never.
- Pretend `dout[0, t, :] = [1, 1]` for all three positions (i.e., upstream gradient is all ones).

Predicted by hand:

- `dwte[0]` = `dout[0,0,:] + dout[0,2,:]` = `[2, 2]`  (token 0 was read at t=0 and t=2)
- `dwte[1]` = `dout[0,1,:]` = `[1, 1]`
- `dwte[2]` = `[0, 0]`  (never read)
- `dwpe[t]` = `dout[0,t,:] = [1, 1]` for each `t`  (each position used once when B=1)

Let's run it and confirm.


In [ ]:
%%writefile course/ch02_build/toy_backward.c
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

void encoder_backward(float* dwte, float* dwpe, float* dout, int* inp,
                      int B, int T, int C) {
    for (int b = 0; b < B; b++)
        for (int t = 0; t < T; t++) {
            float* dout_bt = dout + b*T*C + t*C;
            int ix = inp[b*T + t];
            float* dwte_ix = dwte + ix*C;
            float* dwpe_t  = dwpe + t*C;
            for (int i = 0; i < C; i++) {
                float d = dout_bt[i];
                dwte_ix[i] += d;
                dwpe_t[i]  += d;
            }
        }
}

int main(void) {
    const int B=1, T=3, V=3, C=2;
    int   inp[3]    = {0, 1, 0};
    float dout[1*3*2] = {1,1, 1,1, 1,1};

    float dwte[3*2] = {0};   // V*C, zeroed
    float dwpe[3*2] = {0};   // maxT==T here

    encoder_backward(dwte, dwpe, dout, inp, B, T, C);

    printf("dwte (V=%d rows of size C=%d):\n", V, C);
    for (int v=0; v<V; v++) printf("  dwte[%d] = [%.0f, %.0f]\n", v, dwte[v*C], dwte[v*C+1]);
    printf("dwpe (T=%d rows of size C=%d):\n", T, C);
    for (int t=0; t<T; t++) printf("  dwpe[%d] = [%.0f, %.0f]\n", t, dwpe[t*C], dwpe[t*C+1]);
    return 0;
}


In [ ]:
!gcc -O2 -o course/ch02_build/toy_backward course/ch02_build/toy_backward.c && ./course/ch02_build/toy_backward


You should see exactly:

```
dwte[0] = [2, 2]   <-- token 0 appeared twice
dwte[1] = [1, 1]
dwte[2] = [0, 0]   <-- never appeared
dwpe[0] = [1, 1]
dwpe[1] = [1, 1]
dwpe[2] = [1, 1]
```

The scatter-add pattern is now concrete.


## 10. First Look at OpenMP — Parallel Forward, Racy Backward

The forward pass loops over `B*T` independent positions. Each `(b, t)` writes to its own `out_bt` and only *reads* from `wte` and `wpe`. **Independent writes, shared reads → safe to parallelize.**

OpenMP lets us parallelize a `for` loop with one line:

```c
#pragma omp parallel for collapse(2)
for (int b = 0; b < B; b++) {
    for (int t = 0; t < T; t++) {
        ...
    }
}
```

`#pragma omp parallel for` is a hint to the compiler: *"split the iterations of this loop across threads."* `collapse(2)` says: *"treat the next two nested loops as a single flat loop of length `B*T` for parallelization purposes"* — that gives the runtime more iterations to spread across cores.

You compile with the `-fopenmp` flag, which links the OpenMP runtime. The number of threads defaults to your CPU's logical-core count, or you can set `OMP_NUM_THREADS=N` in the environment.

### The naive backward is racy

What if we add the same `#pragma` to `encoder_backward`? Two threads working on different `(b, t)` positions could end up at the **same row** of `dwte`:

- Thread A: `inp[0,5] = 42` → writes `dwte[42] += d_A`
- Thread B: `inp[1,3] = 42` → writes `dwte[42] += d_B`

Both threads read `dwte[42]`, add their own `d`, and write it back. If their reads/writes interleave, one update overwrites the other and you silently lose gradient. This is a **data race**, and it's the classic parallelism gotcha.

Same problem for `dwpe[t]`: every `b` lane writes to `dwpe[t]` for the same `t`, so threads collide on every position.

The fixes (per-thread accumulators, atomics, or sequential reduction) are real but worth seeing later. **Today's takeaway**: parallelism is not free. Forward and backward of the same op can have wildly different parallel safety profiles.


In [ ]:
%%writefile course/ch02_build/encoder_forward_omp.c
#include <stdio.h>
#include <stdlib.h>
#include <omp.h>

void encoder_forward_omp(float* out, int* inp, float* wte, float* wpe,
                         int B, int T, int C) {
    #pragma omp parallel for collapse(2)
    for (int b = 0; b < B; b++) {
        for (int t = 0; t < T; t++) {
            float* out_bt = out + b*T*C + t*C;
            int ix = inp[b*T + t];
            float* wte_ix = wte + ix*C;
            float* wpe_t  = wpe + t*C;
            for (int i = 0; i < C; i++) out_bt[i] = wte_ix[i] + wpe_t[i];
        }
    }
}

static void* read_file(const char* p, size_t n) {
    FILE* f = fopen(p, "rb"); void* b = malloc(n); size_t r = fread(b,1,n,f); (void)r; fclose(f); return b;
}

int main(int argc, char** argv) {
    int B=atoi(argv[1]), T=atoi(argv[2]), C=atoi(argv[3]), V=atoi(argv[4]), maxT=atoi(argv[5]);
    int*   inp = (int*)   read_file("course/ch02_build/inp.bin",  (size_t)B*T*sizeof(int));
    float* wte = (float*) read_file("course/ch02_build/wte.bin",  (size_t)V*C*sizeof(float));
    float* wpe = (float*) read_file("course/ch02_build/wpe.bin",  (size_t)maxT*C*sizeof(float));
    float* out = (float*) malloc((size_t)B*T*C*sizeof(float));

    printf("OpenMP threads: %d\n", omp_get_max_threads());
    encoder_forward_omp(out, inp, wte, wpe, B, T, C);

    FILE* f = fopen("course/ch02_build/out_omp.bin","wb"); fwrite(out,4,(size_t)B*T*C,f); fclose(f);
    free(inp); free(wte); free(wpe); free(out);
    return 0;
}


In [ ]:
# -fopenmp links the OpenMP runtime
!gcc -O2 -fopenmp -o course/ch02_build/encoder_forward_omp course/ch02_build/encoder_forward_omp.c


In [ ]:
# Re-save inputs (in case earlier cells were skipped) and confirm parallel result matches serial
import numpy as np, torch, subprocess, os
torch.manual_seed(0)
B, T, C, V, maxT = 2, 4, 8, 10, 16

inp = torch.randint(0, V, (B, T), dtype=torch.int32)
wte = torch.randn(V, C); wpe = torch.randn(maxT, C)
inp.numpy().astype(np.int32).tofile("course/ch02_build/inp.bin")
wte.numpy().astype(np.float32).tofile("course/ch02_build/wte.bin")
wpe.numpy().astype(np.float32).tofile("course/ch02_build/wpe.bin")

subprocess.run(["./course/ch02_build/encoder_forward_omp",
                str(B), str(T), str(C), str(V), str(maxT)], check=True)
out_omp = np.fromfile("course/ch02_build/out_omp.bin", dtype=np.float32).reshape(B, T, C)
out_pt  = (wte[inp.long()] + wpe[torch.arange(T)]).numpy()
print(f"max |out_omp - out_pt|: {np.max(np.abs(out_omp - out_pt)):.2e}")


Same answer, multiple threads. With B=2, T=4 the speedup is invisible (the loop is 8 iterations long), but on real GPT-2 sizes (`B=4, T=1024`, so 4096 iterations × 768 channels) parallelizing this loop is a genuine win.


## 11. TODO Exercise 1 — Write `encoder_forward` From Scratch

Boilerplate provided. Fill in the **three pointer expressions** and the **inner-loop body**. The cell below the exercise verifies your output against PyTorch.


In [ ]:
%%writefile course/ch02_build/exercise1.c
#include <stdio.h>
#include <stdlib.h>

void encoder_forward(float* out, int* inp, float* wte, float* wpe,
                     int B, int T, int C) {
    for (int b = 0; b < B; b++) {
        for (int t = 0; t < T; t++) {
            // TODO: row pointer to out[b, t, :]
            float* out_bt = out + b * T * C + t * C;
            // TODO: read the scalar inp[b, t]
            int ix = inp[b * T + t];
            // TODO: row pointer to wte[ix, :]
            float* wte_ix = wte + ix * C;
            // TODO: row pointer to wpe[t, :]
            float* wpe_t  = wpe + t * C;
            for (int i = 0; i < C; i++) {
                // TODO: out_bt[i] = wte_ix[i] + wpe_t[i];
                out_bt[i] = wte_ix[i] + wpe_t[i];
            }
        }
    }
}

static void* rd(const char* p, size_t n){FILE*f=fopen(p,"rb");void*b=malloc(n);size_t r=fread(b,1,n,f);(void)r;fclose(f);return b;}

int main(int argc, char** argv) {
    int B=atoi(argv[1]), T=atoi(argv[2]), C=atoi(argv[3]), V=atoi(argv[4]), maxT=atoi(argv[5]);
    int*   inp = (int*)   rd("course/ch02_build/inp.bin",  (size_t)B*T*sizeof(int));
    float* wte = (float*) rd("course/ch02_build/wte.bin",  (size_t)V*C*sizeof(float));
    float* wpe = (float*) rd("course/ch02_build/wpe.bin",  (size_t)maxT*C*sizeof(float));
    float* out = (float*) calloc((size_t)B*T*C, sizeof(float));
    encoder_forward(out, inp, wte, wpe, B, T, C);
    FILE* f = fopen("course/ch02_build/out_ex1.bin","wb"); fwrite(out,4,(size_t)B*T*C,f); fclose(f);
    free(inp); free(wte); free(wpe); free(out); return 0;
}


In [ ]:
import numpy as np, torch, subprocess
B, T, C, V, maxT = 2, 4, 8, 10, 16
subprocess.run(["gcc","-O2","-o","course/ch02_build/exercise1","course/ch02_build/exercise1.c"], check=True)
subprocess.run(["./course/ch02_build/exercise1", str(B), str(T), str(C), str(V), str(maxT)], check=True)
out_ex = np.fromfile("course/ch02_build/out_ex1.bin", dtype=np.float32).reshape(B, T, C)
torch.manual_seed(0)
_  = torch.randint(0, V, (B, T))     # consume the same RNG state as before
wte = torch.randn(V, C); wpe = torch.randn(maxT, C)
inp_check = np.fromfile("course/ch02_build/inp.bin", dtype=np.int32).reshape(B, T)
out_pt = (wte[torch.from_numpy(inp_check).long()] + wpe[torch.arange(T)]).numpy()
err = np.max(np.abs(out_ex - out_pt))
print(f"max abs diff: {err:.2e}")
print("PASS" if err < 1e-6 else "FAIL — check your offsets and inner loop")


### Solution to Exercise 1

In [ ]:
%%writefile course/ch02_build/exercise1_sol.c
#include <stdio.h>
#include <stdlib.h>

void encoder_forward(float* out, int* inp, float* wte, float* wpe,
                     int B, int T, int C) {
    for (int b = 0; b < B; b++) {
        for (int t = 0; t < T; t++) {
            float* out_bt = out + b*T*C + t*C;
            int   ix     = inp[b*T + t];
            float* wte_ix = wte + ix*C;
            float* wpe_t  = wpe + t*C;
            for (int i = 0; i < C; i++) {
                out_bt[i] = wte_ix[i] + wpe_t[i];
            }
        }
    }
}

static void* rd(const char* p, size_t n){FILE*f=fopen(p,"rb");void*b=malloc(n);size_t r=fread(b,1,n,f);(void)r;fclose(f);return b;}

int main(int argc, char** argv) {
    int B=atoi(argv[1]), T=atoi(argv[2]), C=atoi(argv[3]), V=atoi(argv[4]), maxT=atoi(argv[5]);
    int*   inp = (int*)   rd("course/ch02_build/inp.bin",  (size_t)B*T*sizeof(int));
    float* wte = (float*) rd("course/ch02_build/wte.bin",  (size_t)V*C*sizeof(float));
    float* wpe = (float*) rd("course/ch02_build/wpe.bin",  (size_t)maxT*C*sizeof(float));
    float* out = (float*) calloc((size_t)B*T*C, sizeof(float));
    encoder_forward(out, inp, wte, wpe, B, T, C);
    FILE* f = fopen("course/ch02_build/out_ex1.bin","wb"); fwrite(out,4,(size_t)B*T*C,f); fclose(f);
    free(inp); free(wte); free(wpe); free(out); return 0;
}


In [ ]:
!gcc -O2 -o course/ch02_build/exercise1_sol course/ch02_build/exercise1_sol.c && ./course/ch02_build/exercise1_sol 2 4 8 10 16


## 12. TODO Exercise 2 — Write `encoder_backward`

Same idea, gradient version. Fill in the row-pointer lines and the **two `+=` accumulations** in the inner loop.


In [ ]:
%%writefile course/ch02_build/exercise2.c
#include <stdio.h>
#include <stdlib.h>

void encoder_backward(float* dwte, float* dwpe, float* dout, int* inp,
                      int B, int T, int C) {
    for (int b = 0; b < B; b++) {
        for (int t = 0; t < T; t++) {
            // TODO: row pointer into dout for position (b, t)
            float* dout_bt = dout + b * T * C + t * C;
            // TODO: read the token id at inp[b, t]
            int ix = inp[b * T + t];
            // TODO: row pointer into dwte for token ix
            float* dwte_ix = dwte + ix * C;
            // TODO: row pointer into dwpe for position t
            float* dwpe_t  = dwpe + t * C;
            for (int i = 0; i < C; i++) {
                float d = dout_bt[i];     // TODO: read dout_bt[i]
                // TODO: dwte_ix[i] += d;
                // TODO: dwpe_t[i]  += d;
                dwte_ix[i] += d;
                dwpe_t[i] += d;
            }
        }
    }
}

static void* rd(const char* p, size_t n){FILE*f=fopen(p,"rb");void*b=malloc(n);size_t r=fread(b,1,n,f);(void)r;fclose(f);return b;}

int main(int argc, char** argv) {
    int B=atoi(argv[1]), T=atoi(argv[2]), C=atoi(argv[3]), V=atoi(argv[4]), maxT=atoi(argv[5]);
    int*   inp  = (int*)   rd("course/ch02_build/inp.bin",  (size_t)B*T*sizeof(int));
    float* dout = (float*) rd("course/ch02_build/dout.bin", (size_t)B*T*C*sizeof(float));
    float* dwte = (float*) calloc((size_t)V*C, sizeof(float));
    float* dwpe = (float*) calloc((size_t)maxT*C, sizeof(float));
    encoder_backward(dwte, dwpe, dout, inp, B, T, C);
    FILE* f1=fopen("course/ch02_build/dwte_ex2.bin","wb"); fwrite(dwte,4,(size_t)V*C,f1); fclose(f1);
    FILE* f2=fopen("course/ch02_build/dwpe_ex2.bin","wb"); fwrite(dwpe,4,(size_t)maxT*C,f2); fclose(f2);
    free(inp); free(dout); free(dwte); free(dwpe); return 0;
}


In [ ]:
import numpy as np, torch, subprocess
torch.manual_seed(0)
B, T, C, V, maxT = 2, 4, 8, 10, 16
inp_t = torch.randint(0, V, (B, T))
wte_t = torch.randn(V, C, requires_grad=True)
wpe_t = torch.randn(maxT, C, requires_grad=True)
out = wte_t[inp_t] + wpe_t[torch.arange(T)]
dout = torch.randn_like(out); out.backward(dout)

inp_t.numpy().astype(np.int32).tofile("course/ch02_build/inp.bin")
dout.detach().numpy().astype(np.float32).tofile("course/ch02_build/dout.bin")

subprocess.run(["gcc","-O2","-o","course/ch02_build/exercise2","course/ch02_build/exercise2.c"], check=True)
subprocess.run(["./course/ch02_build/exercise2", str(B), str(T), str(C), str(V), str(maxT)], check=True)

dwte_c = np.fromfile("course/ch02_build/dwte_ex2.bin", dtype=np.float32).reshape(V, C)
dwpe_c = np.fromfile("course/ch02_build/dwpe_ex2.bin", dtype=np.float32).reshape(maxT, C)
err1 = np.max(np.abs(dwte_c - wte_t.grad.numpy()))
err2 = np.max(np.abs(dwpe_c - wpe_t.grad.numpy()))
print(f"dwte max diff: {err1:.2e}")
print(f"dwpe max diff: {err2:.2e}")
print("PASS" if max(err1, err2) < 1e-6 else "FAIL — check your += and pointer expressions")


### Solution to Exercise 2

In [ ]:
%%writefile course/ch02_build/exercise2_sol.c
#include <stdio.h>
#include <stdlib.h>

void encoder_backward(float* dwte, float* dwpe, float* dout, int* inp,
                      int B, int T, int C) {
    for (int b = 0; b < B; b++) {
        for (int t = 0; t < T; t++) {
            float* dout_bt = dout + b*T*C + t*C;
            int   ix       = inp[b*T + t];
            float* dwte_ix = dwte + ix*C;
            float* dwpe_t  = dwpe + t*C;
            for (int i = 0; i < C; i++) {
                float d = dout_bt[i];
                dwte_ix[i] += d;
                dwpe_t[i]  += d;
            }
        }
    }
}

static void* rd(const char* p, size_t n){FILE*f=fopen(p,"rb");void*b=malloc(n);size_t r=fread(b,1,n,f);(void)r;fclose(f);return b;}

int main(int argc, char** argv) {
    int B=atoi(argv[1]), T=atoi(argv[2]), C=atoi(argv[3]), V=atoi(argv[4]), maxT=atoi(argv[5]);
    int*   inp  = (int*)   rd("course/ch02_build/inp.bin",  (size_t)B*T*sizeof(int));
    float* dout = (float*) rd("course/ch02_build/dout.bin", (size_t)B*T*C*sizeof(float));
    float* dwte = (float*) calloc((size_t)V*C, sizeof(float));
    float* dwpe = (float*) calloc((size_t)maxT*C, sizeof(float));
    encoder_backward(dwte, dwpe, dout, inp, B, T, C);
    FILE* f1=fopen("course/ch02_build/dwte_ex2.bin","wb"); fwrite(dwte,4,(size_t)V*C,f1); fclose(f1);
    FILE* f2=fopen("course/ch02_build/dwpe_ex2.bin","wb"); fwrite(dwpe,4,(size_t)maxT*C,f2); fclose(f2);
    free(inp); free(dout); free(dwte); free(dwpe); return 0;
}


In [ ]:
!gcc -O2 -o course/ch02_build/exercise2_sol course/ch02_build/exercise2_sol.c && ./course/ch02_build/exercise2_sol 2 4 8 10 16


## Recap

You now know:

- The embedding layer is **a gather**: `wte + ix*C` is the entire forward in one pointer expression.
- The embedding gradient is **a scatter-add**: gradients accumulate into rows of `dwte` and `dwpe` because the same row is used by many `(b, t)` positions.
- Gradient buffers must be **zeroed before backward** (`gpt2_zero_grad` does this in the real model).
- `#pragma omp parallel for` parallelizes a loop in one line, but **safety depends on what the loop body writes to**. Independent writes → safe (forward). Shared writes → race (naive backward).

### What's next

**Chapter 3 — LayerNorm.** We move from "gather + add" to actual *math*: per-row mean and variance, the fused normalize-scale-shift, and the famous `rstd` cache trick that makes the backward pass tractable. We'll also see why the backward of LayerNorm is much hairier than the forward — and how to derive it cleanly from the chain rule.
